# Agent Action Guardrails

Level 5: Senior FDE / Staff-Level Scenarios.

**Problems in this notebook**

- Problem 43: Agent Tool Execution Safety
- Problem 47: Human-in-the-Loop Approval Queue
- Problem 48: Tool-Call Retry Prevention

**How to use this notebook**

1. Read the problem and say your plan out loud before you write any code.
2. Use the AI prompt given (or your own narrow one) to fill the starter. Read the AI's code before you run it.
3. Run the checks. Open the reference only after you have tried.

The ideas behind these problems are in `01_agent_action_guardrails_THEORY.md`.

In [ ]:
def run_checks(fn, cases):
    """cases: list of (args_tuple, expected). Prints a short report, never raises."""
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*args)
        except NotImplementedError:
            print("Not implemented yet. Fill in the starter first.")
            return False
        except Exception as e:
            print(f"Case {i}: raised {type(e).__name__}: {e}")
            continue
        if got == expected:
            passed += 1
        else:
            print(f"Case {i}: expected {expected!r}, got {got!r}")
    print(f"{passed}/{len(cases)} checks passed")
    return passed == len(cases)

## Problem 43: Agent Tool Execution Safety

**The situation**

An AI agent in a customer's support platform proposes tool calls such as `search_kb`, `update_crm` and `delete_tenant`. Some tools only read data, others change the customer's systems. The model suggests the call, but the platform must decide if the user is allowed to run it.

**Your task**

Write `can_execute_agent_tool(tool, user_roles)` that returns `True` only when the user's roles are enough for the tool's risk level. Anything unclear is a no.

**Example**

`{"name": "update_crm", "risk": "write"}`, roles `{"viewer"}` -> `False`

`{"name": "update_crm", "risk": "write"}`, roles `{"operator"}` -> `True`

**Ask before coding**

- Where does the risk label come from: our tool registry or the model's output?
- Which roles may run write tools and admin tools?
- Should a denied attempt be logged or sent for approval?

**Plan to say aloud**

1. Keep a small table: risk level -> set of roles allowed.
2. Look up the tool's risk. Missing or unknown risk means deny.
3. Allow only if the user has at least one allowed role.
4. Say that the risk comes from our registry, never from the model.

### Prompt for your AI assistant

```text
Write a Python function can_execute_agent_tool(tool: dict, user_roles: set[str]) -> bool.
Rules: tool["risk"] is "read", "write" or "admin".
read is allowed for viewer, operator or admin. write is allowed for operator or admin.
admin is allowed only for admin. A missing or unknown risk returns False.
No roles returns False. Use a dict from risk to allowed roles. Standard library only.
Show only the function.
```

**Check the AI's code for**

- Defaults a missing risk to "read". That fails open: an unlabeled tool becomes runnable.
- Returns `True` for an unknown risk when the user is admin. Unknown should still be a no.
- Uses `bool(user_roles)` for read, so any made-up role string passes.
- Trusts a `risk` or `allowed` field that the model wrote into its own tool call.

In [ ]:
def can_execute_agent_tool(tool: dict, user_roles: set[str]) -> bool:
    """Return True only if user_roles are enough for tool["risk"].

    Rules:
    - read: viewer, operator or admin
    - write: operator or admin
    - admin: admin only
    - missing or unknown risk -> False
    - no roles -> False
    """
    raise NotImplementedError

In [ ]:
cases = [
    (({"name": "search_kb", "risk": "read"}, {"viewer"}), True),
    (({"name": "update_crm", "risk": "write"}, {"viewer"}), False),
    (({"name": "update_crm", "risk": "write"}, {"operator"}), True),
    (({"name": "delete_tenant", "risk": "admin"}, {"operator"}), False),
    (({"name": "mystery_tool"}, {"admin"}), False),           # missing risk: deny
    (({"name": "search_kb", "risk": "read"}, set()), False),  # no roles: deny
]
run_checks(can_execute_agent_tool, cases)

### Reference solution

Try first. Then compare.

In [ ]:
ALLOWED_ROLES = {
    "read": {"viewer", "operator", "admin"},
    "write": {"operator", "admin"},
    "admin": {"admin"},
}


def can_execute_agent_tool_reference(tool: dict, user_roles: set[str]) -> bool:
    allowed = ALLOWED_ROLES.get(tool.get("risk"))
    if allowed is None:  # missing or unknown risk: fail closed
        return False
    return bool(allowed & set(user_roles))


assert run_checks(can_execute_agent_tool_reference, cases)

**Explain it in 30 seconds**

> "The agent can propose a tool call, but the platform decides if it runs. I keep one table from risk level to allowed roles and look the tool up in it. If the risk is missing or unknown, I say no. The check runs before the tool, so a bad call never touches the customer's system."

**Edge cases to mention**

- Tool with no risk label, or a new label like `"destructive"`.
- User with no roles, or roles in odd casing like `"Admin"`.
- A tenant policy that blocks write tools even for operators.
- A high-risk tool that should go to a human instead of a plain yes or no.

**If they ask "how would this work in production?"**

- Read the risk from a server-side tool registry, never from the model's output.
- Log every denied attempt with the user, tenant and tool, so security can review it.
- Give tools scoped credentials, so even an allowed call can only touch what it needs.

## Problem 47: Human-in-the-Loop Approval Queue

**The situation**

A finance team uses an AI agent that can issue refunds and change vendor bank details. Low-risk actions may run on their own. Risky actions must wait for a human, and an approval only counts for a short time and must come from someone other than the requester.

**Your task**

Write `decide_action(action, now, approval_ttl=900)` that returns `"auto_execute"`, `"execute_approved"` or `"approval_required"`. `now` is passed in as seconds, so the check is easy to test.

**Example**

`{"risk": "low"}`, now=100 -> `"auto_execute"`

`{"risk": "critical"}`, now=100 -> `"approval_required"`

`{"risk": "high", "requested_by": "agent", "approval": {"approved_by": "maria", "approved_at": 50}}`, now=100 -> `"execute_approved"`

**Ask before coding**

- Which risk levels may run without a human?
- How long does an approval stay valid?
- Can the person who asked for the action also approve it?

**Plan to say aloud**

1. If risk is low or medium, auto-execute.
2. Everything else, including missing or unknown risk, needs an approval.
3. An approval counts only if it has an approver, the approver is not the requester, and it is not expired.
4. Otherwise, send it to the approval queue.

### Prompt for your AI assistant

```text
Write a Python function decide_action(action: dict, now: int, approval_ttl: int = 900) -> str.
Rules: if action["risk"] is "low" or "medium", return "auto_execute".
Any other risk, including missing, needs approval. action.get("approval") may hold
{"approved_by": str, "approved_at": int}. Return "execute_approved" only if approved_by is set,
approved_by != action.get("requested_by"), and 0 <= now - approved_at <= approval_ttl.
Otherwise return "approval_required". Do not call time.time(); use the now argument.
Show only the function.
```

**Check the AI's code for**

- Lists only `high` and `critical` as risky, so a missing or misspelled risk auto-executes.
- Forgets the expiry check, so an approval from last month still works.
- Lets the requester approve their own action.
- Calls `time.time()` inside the function, which makes the checks flaky.

In [ ]:
def decide_action(action: dict, now: int, approval_ttl: int = 900) -> str:
    """Route an agent action.

    Rules:
    - risk "low" or "medium" -> "auto_execute"
    - any other risk (high, critical, missing, unknown) needs approval:
      "execute_approved" if action["approval"] has approved_by (not the requester)
      and 0 <= now - approved_at <= approval_ttl, else "approval_required"
    """
    raise NotImplementedError

In [ ]:
cases = [
    (({"id": "a1", "risk": "low"}, 100), "auto_execute"),
    (({"id": "a2", "risk": "critical"}, 100), "approval_required"),
    (({"id": "a3"}, 100), "approval_required"),  # missing risk: needs a human
    (({"id": "a4", "risk": "high", "requested_by": "agent",
       "approval": {"approved_by": "maria", "approved_at": 50}}, 100), "execute_approved"),
    (({"id": "a5", "risk": "high", "requested_by": "agent",
       "approval": {"approved_by": "maria", "approved_at": 0}}, 1000), "approval_required"),  # expired
    (({"id": "a6", "risk": "high", "requested_by": "maria",
       "approval": {"approved_by": "maria", "approved_at": 90}}, 100), "approval_required"),  # self-approval
]
run_checks(decide_action, cases)

### Reference solution

Try first. Then compare.

In [ ]:
AUTO_RISKS = {"low", "medium"}


def decide_action_reference(action: dict, now: int, approval_ttl: int = 900) -> str:
    if action.get("risk") in AUTO_RISKS:
        return "auto_execute"
    approval = action.get("approval") or {}
    approver = approval.get("approved_by")
    approved_at = approval.get("approved_at")
    if (
        approver
        and approver != action.get("requested_by")
        and approved_at is not None
        and 0 <= now - approved_at <= approval_ttl
    ):
        return "execute_approved"
    return "approval_required"


assert run_checks(decide_action_reference, cases)

**Explain it in 30 seconds**

> "I list the risk levels that are safe to run alone, and everything else needs a human. That way a missing or wrong label goes to the queue, not straight to execution. An approval only counts if it is recent and came from someone other than the requester. The time is passed in, so the rule is easy to test."

**Edge cases to mention**

- Missing risk label, or a label the code has never seen.
- An approval that is exactly at the time limit.
- The requester approving their own action.
- A sensitive action that the model labeled as low risk.

**If they ask "how would this work in production?"**

- Store the full context with each approval: who asked, who approved, when, and the exact payload.
- Set the risk label from our own tool registry, not from the model.
- Bind the approval to a hash of the payload, so a changed action needs a new approval.

## Problem 48: Tool-Call Retry Prevention

**The situation**

An agent calls `create_ticket` in a customer's helpdesk. The network drops before the reply comes back, so the agent retries. Without a guard, the customer gets two tickets, two emails, or two CRM updates.

**Your task**

Write `should_execute_tool_call(tenant_id, idempotency_key, seen, now, ttl_seconds=3600)`. Return `True` and record the key the first time. Return `False` for a repeat of the same tenant and key inside the time window. `seen` is a dict you update; `now` is passed in.

**Example**

Calls `("t1", "k1", now=0)` then `("t1", "k1", now=5)` -> `True`, `False`

Calls `("t1", "k1", now=0)` then `("t2", "k1", now=5)` -> `True`, `True` (different tenant)

**Ask before coding**

- Who makes the idempotency key, and what goes into it?
- Should two tenants with the same key block each other?
- How long should we remember a key?

**Plan to say aloud**

1. Build a scoped key from tenant and idempotency key.
2. If we saw it inside the time window, return False.
3. Otherwise record it with the current time and return True.
4. Say what happens if the call later fails.

### Prompt for your AI assistant

```text
Write a Python function should_execute_tool_call(tenant_id: str, idempotency_key: str,
seen: dict, now: int, ttl_seconds: int = 3600) -> bool.
Rules: the store key is the tuple (tenant_id, idempotency_key). seen maps that tuple to the
time it was first recorded. If the key is in seen and now - seen[key] < ttl_seconds, return False.
Otherwise set seen[key] = now and return True. Do not use time.time() or sleep.
Show only the function.
```

**Check the AI's code for**

- Uses only the idempotency key, so tenant B is blocked by tenant A's key.
- Records the key but forgets to return `True`, or checks after writing, so the first call is refused.
- Ignores the time window, so the store grows forever and keys never expire.
- Creates a new `seen` store inside the function, so nothing is remembered between calls.

In [ ]:
def should_execute_tool_call(tenant_id: str, idempotency_key: str, seen: dict,
                             now: int, ttl_seconds: int = 3600) -> bool:
    """Return True the first time a (tenant_id, idempotency_key) is seen, and record it.

    Rules:
    - store key is (tenant_id, idempotency_key); seen maps it to the time first recorded
    - seen before and now - seen[key] < ttl_seconds -> False
    - otherwise set seen[key] = now and return True
    """
    raise NotImplementedError


def replay(fn, calls):
    """Run calls [(tenant_id, key, now), ...] against one fresh store. Returns the list of decisions."""
    seen = {}
    return [fn(tenant, key, seen, now) for tenant, key, now in calls]

In [ ]:
cases = [
    (([("t1", "k1", 0), ("t1", "k1", 5)],), [True, False]),     # retry is blocked
    (([("t1", "k1", 0), ("t2", "k1", 5)],), [True, True]),      # keys are tenant-scoped
    (([("t1", "k1", 0), ("t1", "k2", 5)],), [True, True]),      # new key runs
    (([("t1", "k1", 0), ("t1", "k1", 4000)],), [True, True]),   # window expired
    (([],), []),
]
run_checks(lambda calls: replay(should_execute_tool_call, calls), cases)

### Reference solution

Try first. Then compare.

In [ ]:
def should_execute_tool_call_reference(tenant_id: str, idempotency_key: str, seen: dict,
                                       now: int, ttl_seconds: int = 3600) -> bool:
    scoped_key = (tenant_id, idempotency_key)
    first_seen = seen.get(scoped_key)
    if first_seen is not None and now - first_seen < ttl_seconds:
        return False
    seen[scoped_key] = now
    return True


assert run_checks(lambda calls: replay(should_execute_tool_call_reference, calls), cases)

**Explain it in 30 seconds**

> "Every write tool call carries an idempotency key, which is a unique id for one intended action. I scope it by tenant, check if I have seen it recently, and only run the call the first time. A retry after a network error then does nothing instead of creating a duplicate. Time is passed in, so I can test the expiry without waiting."

**Edge cases to mention**

- Same key from two different tenants.
- Same key sent with a different payload, which is likely a bug in the caller.
- The first call failed: should a retry be allowed?
- Two retries arriving at the same moment on two servers.

**If they ask "how would this work in production?"**

- Use an atomic store, like a database unique constraint or Redis `SET NX` with a TTL, so two servers cannot both win.
- Save the payload hash and the result, so a retry gets the first answer back instead of a bare no.
- Mark failed calls clearly so a safe retry is allowed, and pass the key to the target system when it supports one.

## Wrap-up

- The model proposes; the platform decides. Every check here runs before the tool touches a customer system.
- When something is missing or unknown, the safe answer is no, or ask a human.
- State and time are passed in (`seen`, `now`), so the rules are easy to test without real clocks or sleeps.